# Run Three-Version Benchmark

**Purpose:**
- Use the T112A shared runner to run selected dataset rows through baseline, dflash, and cc-dflash.
- Safely manage configuration so execution doesn't unexpectedly load models without permission.

In [ ]:
import os, sys, json
from pathlib import Path

# Ensure repository root safe path
ROOT = Path.cwd()
if not (ROOT / "src/ccdf").exists():
    ROOT = ROOT.parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
os.chdir(ROOT)

print(f"Project root: {ROOT}")


## Configuration

In [ ]:
DATASET = "gsm8k"           # gsm8k or qmsum
LIMIT = 3
SEED = 42
MAX_NEW_TOKENS = 128
CONDITIONS = [
    "baseline_ar",
    "dflash_r1",
    "cc_dflash_r2",
]
RUN_REAL_MODELS = False
if 'CCDF_NOTEBOOK_REAL_RUN' in os.environ:
    RUN_REAL_MODELS = os.environ['CCDF_NOTEBOOK_REAL_RUN'] == '1'

RESUME = True
OUTPUT_ROOT = "results/charts/notebook_demo"


## Execute Models

In [ ]:
from ccdf.demo import DemoRunner
from ccdf.demo.contracts import RunRequest
from ccdf.demo.writers import write_jsonl_append, write_flat_csv_append, write_json
from ccdf.demo.adapters.gsm8k import gsm8k_row_to_request
from ccdf.demo.adapters.qmsum import qmsum_row_to_request
from ccdf.config import load_config

if not RUN_REAL_MODELS:
    print("RUN_REAL_MODELS is False. Running in dry-run mode.")
    print("To run with real models, set RUN_REAL_MODELS = True or CCDF_NOTEBOOK_REAL_RUN=1")
    config = {"dry_run": True}
else:
    try:
        config = load_config("config.yml")
    except FileNotFoundError:
        config = {}
    if isinstance(config, dict):
        class ConfigDict(dict): pass
        config = ConfigDict(config)
    config.dry_run = False

runner = DemoRunner(config)

eval_path = ROOT / "data/evaluation" / f"{DATASET}_test.jsonl"
if not eval_path.exists():
    print(f"Data not found at {eval_path}. Run Notebook 1 first to prepare the dataset.")
    if not RUN_REAL_MODELS:
        print("Using a synthetic dummy row to preview the pipeline.")
        rows = [{"id": "mock_1", "context": "Mock context", "question": "Mock question?", "expected_answer": "Mock", "split": "test"}]
    else:
        raise FileNotFoundError(f"Missing data: {eval_path}")
else:
    with open(eval_path, "r", encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]
rows = rows[:LIMIT]

out_dir = ROOT / OUTPUT_ROOT
runs_dir = out_dir / "runs"
tables_dir = out_dir / "tables"
summaries_dir = out_dir / "summaries"
manifests_dir = out_dir / "manifests"

runs_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)
summaries_dir.mkdir(parents=True, exist_ok=True)
manifests_dir.mkdir(parents=True, exist_ok=True)

out_jsonl = runs_dir / f"{DATASET}_three_version.jsonl"
out_csv = tables_dir / f"{DATASET}_three_version.csv"

skip_count = 0
if RESUME and out_jsonl.exists():
    with open(out_jsonl, "r", encoding="utf-8") as f:
        skip_count = sum(1 for line in f if line.strip()) // len(CONDITIONS)
    print(f"Resuming after skipping {skip_count} completed rows.")
elif out_jsonl.exists():
    out_jsonl.unlink()
    if out_csv.exists(): out_csv.unlink()

for i, row in enumerate(rows):
    if i < skip_count: continue
    print(f"Processing row {i+1}/{len(rows)}")
    for cond in CONDITIONS:
        try:
            if DATASET == "gsm8k":
                req = gsm8k_row_to_request(row, cond, seed=SEED, max_new_tokens=MAX_NEW_TOKENS)
            else:
                req = qmsum_row_to_request(row, cond, seed=SEED, max_new_tokens=MAX_NEW_TOKENS)
            
            print(f"  Condition: {cond} | Profile: {req.prompt_profile}")
            res = runner.run(req)
            
            write_jsonl_append(res, out_jsonl)
            write_flat_csv_append(res, out_csv)
            
        except Exception as e:
            print(f"  Failed {cond}: {e}")

summary = {
    "dataset": DATASET,
    "limit": LIMIT,
    "conditions_run": CONDITIONS,
    "jsonl_path": str(out_jsonl.relative_to(ROOT)),
    "csv_path": str(out_csv.relative_to(ROOT))
}
write_json(summary, summaries_dir / f"{DATASET}_three_version_summary.json", overwrite=True)
write_json({"status": "completed", "dataset": DATASET}, manifests_dir / "notebook_run_manifest.json", overwrite=True)
print("Finished!")


## Interpretation Notes
- For GSM8K, the output may show a numeric exact match as a proxy for correctness.
- For QMSum, **Semantic correctness is not claimed; quality fields are bounded proxies only.**